#Initializaiton

In [0]:
%sql
USE CATALOG salesdatalakehouse;

## Read from Silver Schema, Prepare and Serve in Gold Schema

###Prepare dimension customers

In [0]:
display(
  spark.sql(
    '''
      CREATE VIEW gold.dim_customers AS
      SELECT 
        ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
        ci.customer_id,
        ci.customer_key AS customer_number,
        ci.first_name,
        ci.last_name,
        la.country,
        ci.marital_status,
        CASE WHEN ci.gender != 'n/a' THEN ci.gender ELSE COALESCE(ca.gender, 'n/a') END AS gender,
        ca.birth_date,
        ci.create_date
      FROM silver.crm_cust_info ci
      LEFT JOIN silver.erp_cust_az12 ca
      ON ci.customer_key = ca.customer_id
      LEFT JOIN silver.erp_loc_a101 la
      ON ci.customer_key = la.customer_id
    '''
  )
)

###Prepare dimension products

In [0]:
display(
    spark.sql(
      '''
        CREATE VIEW gold.dim_products AS
        SELECT 
            ROW_NUMBER() OVER (ORDER BY pi.start_date, pi.product_key) AS product_key,
            pi.product_id,
            pi.product_key AS product_number,
            pi.product_name,
            pi.category_id,
            pc.category,
            pc.subcategory,
            pc.maintenance,
            pi.product_cost,
            pi.product_line,
            pi.start_date            
        FROM silver.crm_prd_info pi
        LEFT JOIN silver.erp_px_cat_g1v2 pc
        ON pi.category_id = pc.category_id
        WHERE pi.end_date IS NULL --Filter out Historical data
      '''
    )
)

###Prepare fact sales

In [0]:
display(
    spark.sql(
    '''
        CREATE VIEW gold.fact_sales AS
        SELECT 
            sd.order_number,
            p.product_key,
            c.customer_key,
            sd.order_date,
            sd.ship_date,
            sd.due_date,
            sd.sales_amount,
            sd.quantity,
            sd.price
        FROM silver.crm_sales_details sd
        LEFT JOIN gold.dim_customers c
        ON sd.customer_id = c.customer_id
        LEFT JOIN gold.dim_products p
        ON sd.product_key = p.product_number
    '''
))